Groundwater | Flow Modeling Track

# Step 5: Model Calibration – From Observations to Parameters

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → **5-Calibrate** → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 4 you built a MODFLOW 6 model of the Limmat Valley aquifer with a Voronoi (DISV) grid and uniform hydraulic conductivity (K = 20 m/d). The model runs, but we haven’t checked whether it actually reproduces the groundwater levels we observe in the field. That’s what calibration does — and we’ll discover along the way that we need more data and more flexible parameterisation than we started with.

| **Core content** | ~60 minutes |
|:--|:--|
| **Optional: Exercises & advanced topics** | +20 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Assess** observation data coverage and identify spatial gaps in monitoring networks
2. **Evaluate** a pumping test using the Cooper-Jacob straight-line method
3. **Calculate** calibration metrics (ME, MAE, RMSE, R², NRMSE) and interpret their meaning
4. **Explain** how PEST++ with pilot points estimates spatially varying parameters
5. **Interpret** a calibrated hydraulic conductivity field and the role of prior information
6. **Verify** calibration quality through water balance checks and residual analysis

## Prerequisites

Before starting this notebook, you should have:
- **Completed [4_model_implementation.ipynb](4_model_implementation.ipynb)** — your model must run successfully
- Basic understanding of **Darcy’s Law** and the concept of hydraulic conductivity
- Familiarity with the **Limmat Valley case study** from the earlier notebooks

In [ ]:
# Setup
import sys
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.mf6 import MFSimulation

from IPython.display import display, HTML

# Add support modules to path
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

from data_utils import (
    download_named_file,
    get_default_data_folder,
)
from model_io_utils import load_base_simulation, load_model_boundary, format_budget_summary
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

import pumping_test_utils as ptu
import calibration_utils as cu

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

## Introduction

**Calibration** is the process of adjusting model parameters so that simulated outputs match field observations as closely as possible. Without calibration, a model is simply a hypothesis — calibration tests that hypothesis against reality.

In Step 4 we assigned a uniform hydraulic conductivity of K = 20 m/d. But how do we know that value is correct? The answer is: we don’t — not until we compare the model’s predictions to actual measurements.

This notebook follows a natural problem-solving story:

1. **Load the model** and see what observation data we have → we discover we have very few wells
2. **Run the uncalibrated model** and check the fit → it may look OK locally, but we have no data elsewhere
3. **Analyse a pumping test** to get an independent K estimate in a data-sparse area
4. **Calibrate with PEST++** using pilot points and the pumping-test K as prior information
5. **Assess the calibrated model** — does it make physical sense?

---

## 1 - Load Model and Observation Data

We start by loading the MODFLOW 6 model from Step 4 and copying it to a separate **calibration workspace** (so we don’t modify the original model files).

In [ ]:
# --- 1. Define paths ---
DATA_DIR = get_default_data_folder()

# The model from Notebook 4 lives here:
sim_name = 'limmat_valley'
nb4_workspace = os.path.join(DATA_DIR, 'notebook4_model')

# We'll work in a separate calibration workspace to keep the original model intact
calib_workspace = os.path.join(DATA_DIR, 'calibration')

# Always start fresh: copy the NB4 model to the calibration workspace
if not os.path.exists(os.path.join(nb4_workspace, 'mfsim.nam')):
    raise FileNotFoundError(
        f"Notebook 4 model not found at {nb4_workspace}\n"
        "Please run Notebook 4 first to generate the MODFLOW 6 model."
    )

if os.path.exists(calib_workspace):
    shutil.rmtree(calib_workspace)
shutil.copytree(nb4_workspace, calib_workspace)
print(f"Fresh calibration workspace created: {calib_workspace}")

workspace = calib_workspace

In [ ]:
# --- 2. Load the MODFLOW 6 simulation ---
sim = load_base_simulation(workspace)
gwf = sim.get_model(sim_name)

modelgrid = gwf.modelgrid
modelgrid.set_coord_info(crs="EPSG:2056")

# Get idomain for masking
disv = gwf.get_package('DISV')
idomain = disv.idomain.array

# Clean up any WEL entries in inactive cells (can occur from NB4 sensitivity runs)
wel = gwf.get_package('WEL')
if wel is not None:
    wd = wel.stress_period_data.get_data(0)
    if wd is not None and len(wd) > 0:
        id_flat = idomain.flatten()
        keep = [r for r in wd if id_flat[r['cellid'][-1]] > 0]
        n_removed = len(wd) - len(keep)
        if n_removed > 0:
            import numpy.lib.recfunctions as rfn
            wel.stress_period_data.set_data(keep, key=0)
            print(f"Removed {n_removed} WEL entries in inactive cells.")

print(f"Grid type: {modelgrid.grid_type}")
print(f"Number of cells per layer: {modelgrid.ncpl}")
print(f"Active cells: {(idomain.flatten() > 0).sum()}")

In [ ]:
# --- 3. Load model boundary for spatial filtering ---
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

### 1.1 Load AWEL observation data

The Canton of Zürich (AWEL) maintains a network of groundwater monitoring wells with long-term head measurements. Let's load this data and see which wells fall within our model domain.

In [ ]:
# --- 4. Load AWEL observation wells ---
# load_awel_observations handles:
#   - Reading the CSV time series
#   - Converting LV03 → LV95 coordinates
#   - Computing mean heads per well
#   - Filtering to wells within the model boundary
obs_gdf = cu.load_awel_observations(DATA_DIR, boundary_gdf, use_mean=True)

print(f"Observation wells within model domain: {len(obs_gdf)}")
print(cu.get_observation_summary(obs_gdf))
display(obs_gdf[['obs_id', 'x', 'y', 'head_m']])

### 1.2 Generate synthetic observations

Since the real AWEL wells are clustered at the eastern end of the aquifer, we supplement them with **synthetic observation points** to provide spatial coverage for the calibration demonstration.

The synthetic observations are sampled from a **reference model** with spatially varying K that reflects the geological heterogeneity of the alluvial aquifer. In the Limmat Valley, aquifer thickness increases from west (~10 m) to east (~50 m), and deeper sections tend to contain coarser gravels with higher K. The reference K field captures this thickness-dependent trend plus random spatial variability, producing head patterns more realistic than a uniform-K model.

In [ ]:
# --- 5. Generate synthetic observations for better spatial coverage ---
# Real AWEL wells are clustered — we add synthetic points for teaching purposes

# Generate a reference K field that varies with aquifer thickness
# (deeper zones = coarser gravels = higher K, reflecting alluvial geology)
top = disv.top.array.flatten()
botm = disv.botm.array.flatten()

k_reference = cu.generate_reference_k_field(
    modelgrid, idomain, top, botm, seed=42,
)

# Run the model with the reference K field to get realistic synthetic heads
gwf.npf.k.set_data(k_reference)
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("Reference model run failed — check the listing file.")

head_ref = gwf.output.head().get_data()

# Restore uniform K=20 for the uncalibrated assessment in Section 2
gwf.npf.k.set_data(20.0)

# Get river cell indices to exclude from synthetic observation placement
riv = gwf.get_package('RIV')
if riv is not None:
    riv_data = riv.stress_period_data.get_data(0)
    riv_cells = np.array([r[1] for r in riv_data]) if riv_data is not None else np.array([])
else:
    riv_cells = np.array([])

# Load river polygons for spatial filtering (north/south classification + buffer)
gis_dir = os.path.join(os.path.dirname(boundary_path), '')  # same folder as boundary
river_gis_path = os.path.join(os.path.dirname(boundary_path), 'AV_Gewasser_-OGD.gpkg')
river_all = gpd.read_file(river_gis_path)
river_gdf = river_all[
    river_all['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
    & river_all.intersects(boundary_gdf.geometry.iloc[0])
]

synth_gdf = cu.generate_synthetic_observations(
    modelgrid, head_ref, idomain,
    n_points=5, noise_std=1.3, seed=42,
    exclude_cells=riv_cells,
    river_gdf=river_gdf,
    boundary_polygon=boundary_gdf.geometry.iloc[0],
    avoid_boundaries_m=100,
    exclude_north_of_river=True,
)

# Combine real + synthetic observations
obs_gdf = cu.combine_observations(obs_gdf, synth_gdf)
print(cu.get_observation_summary(obs_gdf))

In [ ]:
# --- 6. Display combined observation dataset ---
display(obs_gdf[['obs_id', 'x', 'y', 'head_m', 'is_synthetic', 'source']].reset_index(drop=True))

In [ ]:
# Checkpoint 1: How many total observation points do you have?
check_task_with_solution('task05_checkpoint_1')

### 1.3 Visualise the observation network

In [ ]:
# --- 7. Map of observation wells on the model grid ---
fig = cu.plot_observation_network(
    obs_gdf,
    boundary_gdf=boundary_gdf,
    title='Observation Network — AWEL + Synthetic Wells',
)
plt.show()

> 💡 **Key observation:** The 4 real AWEL wells are clustered at the eastern end of the aquifer. Without the synthetic supplements, there would be **no observation data** in the central or downstream portions. Even with synthetic points, only 4 independent measurements constrain the model — the rest are artificial.

> **Think about it:** If we calibrate uniform K to match heads at a few nearby wells, how confident can we be in the model's predictions 2 km downstream?

---

## 2 - Initial Model Assessment

Let’s run the model with its current parameterisation (uniform K = 20 m/d from Notebook 4) and see how well it matches the observations.

In [ ]:
# --- 1. Run the model ---
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)

if success:
    print("Model run completed successfully.")
else:
    print("Model run FAILED. Check the listing file for errors.")

In [ ]:
# --- 2. Load simulated heads ---
head = gwf.output.head().get_data()
heads_flat = head.flatten() if head.ndim > 1 else head

print(f"Head array shape: {head.shape}")
valid_heads = heads_flat[heads_flat > -1e10]
print(f"Head range: {valid_heads.min():.2f} to {valid_heads.max():.2f} m a.s.l.")

In [ ]:
# --- 3. Extract simulated heads at observation wells ---
sim_heads = cu.extract_heads_at_observations(head, obs_gdf, modelgrid)
obs_gdf['simulated_head'] = sim_heads

# Remove any wells that couldn't be matched to grid cells
valid_mask = ~np.isnan(sim_heads)
obs_valid = obs_gdf[valid_mask].copy()

metrics_initial = cu.calculate_calibration_metrics(
    obs_valid['head_m'].values, obs_valid['simulated_head'].values,
)
print(cu.format_metrics_table(metrics_initial))

In [ ]:
# Checkpoint 2: What is the initial RMSE (m)?
check_task_with_solution('task05_checkpoint_2')

In [ ]:
# --- 4. Observed vs. simulated scatter plot ---
fig = cu.plot_calibration_scatter(
    obs_valid['head_m'].values,
    obs_valid['simulated_head'].values,
    obs_gdf=obs_valid,
    title='Initial (Uncalibrated) Model',
)
plt.show()

In [ ]:
# --- 5. Residual map ---
residuals = obs_valid['simulated_head'].values - obs_valid['head_m'].values
fig = cu.plot_residual_map(
    obs_valid, residuals,
    boundary_gdf=boundary_gdf,
    title='Residual Map — Uncalibrated Model',
)
plt.show()

### Discussion

With only a few real observation points — all in a single cluster — we could adjust K to fit them reasonably well. But we have **very limited information** about the rest of the domain. The synthetic observations help us assess model performance more broadly, but they are derived from the model itself.

This motivates two actions:
1. **Get more data** — perform a pumping test in a data-sparse area
2. **Allow K to vary spatially** — because the pumping test may reveal different K values

---

## 3 - Pumping Test Evaluation

To get an independent estimate of K in the central part of the aquifer — where we have no observation wells — we performed a **pumping test**. A well was pumped at a constant rate while drawdown was monitored at 4 observation wells at different distances.

We analyse this test using the **Cooper-Jacob straight-line method**.

### 3.1 Cooper-Jacob Theory Reminder

<details>
<summary><strong>Click to expand: Cooper-Jacob equations</strong></summary>

The **Theis solution** describes drawdown around a pumping well in a confined, infinite aquifer:

$$s(r,t) = \frac{Q}{4\pi T}\, W(u) \qquad \text{where} \quad u = \frac{r^2 S}{4Tt}$$

For **late times** when $u < 0.01$, the Cooper-Jacob approximation gives a **straight line** on a semi-log plot ($s$ vs. $\log_{10} t$):

$$s \approx \frac{2.3\, Q}{4\pi T} \log_{10}\left(\frac{2.25\, T\, t}{r^2 S}\right)$$

From the slope $\Delta s$ (drawdown per log cycle) and the zero-intercept $t_0$:

$$T = \frac{2.3\, Q}{4\pi\, \Delta s} \qquad S = \frac{2.25\, T\, t_0}{r^2} \qquad K = \frac{T}{b}$$

**Key insight:** The slope depends only on $Q$ and $T$, not on distance $r$. All observation wells should yield the **same $T$**.

</details>

### 3.2 Load pumping test data

In [ ]:
# --- 1. Generate pumping test data ---
# (Reproducible synthetic data from the Theis solution + measurement noise)
from generate_pumping_test_data import generate_pumping_test_data

pt_output_dir = os.path.join(workspace, 'pumping_test')
os.makedirs(pt_output_dir, exist_ok=True)
csv_path, json_path = generate_pumping_test_data(output_dir=pt_output_dir)

# Load the data
pt_data = pd.read_csv(csv_path)
with open(json_path) as f:
    pt_meta = json.load(f)

# Given information
Q = pt_meta['pumping_rate_m3_per_day']   # pumping rate [m³/d]
b = pt_meta['aquifer_thickness_m']        # aquifer thickness [m]

print(f"Pumping test: {len(pt_data)} measurements, "
      f"{pt_data['well_id'].nunique()} observation wells")
print(f"Q = {Q:.0f} m³/d ({Q/86.4:.1f} L/s),  b = {b:.0f} m")
print()
for wid, grp in pt_data.groupby('well_id'):
    r = grp['distance_m'].iloc[0]
    print(f"  {wid}: r = {r:.0f} m, {len(grp)} points")

### 3.3 Raw drawdown data

In [ ]:
fig, ax = ptu.plot_all_wells_raw(pt_data)
plt.show()

Closer wells (OW-1, OW-2) show larger drawdown and respond faster. Farther wells respond more slowly. All curves approach straight lines on a semi-log plot — the **Cooper-Jacob regime**.

### 3.4 Cooper-Jacob Analysis — Step by Step

> ✏️ **Exercise:** Work through the Cooper-Jacob method step by step, starting with observation well **OW-2** (r = 30 m).

**Step 1 — Fit the straight line**

Use `ptu.cooper_jacob_fit()` to fit the late-time straight line, then `ptu.plot_semilog()` to visualise the fit. Read off the **slope** $\Delta s$ (drawdown per log cycle).

```python
# Function signatures:
ptu.cooper_jacob_fit(time_min, drawdown_m)          → dict with 'slope', 't0_min', ...
ptu.plot_semilog(time, drawdown, fit, well_id, r_m) → ax
```

In [ ]:
# --- Step 1: Cooper-Jacob fit for OW-2 ---

# Extract OW-2 data (provided)
ow2 = pt_data[pt_data['well_id'] == 'OW-2']
r_ow2 = ow2['distance_m'].iloc[0]

# ✏️ Fit the Cooper-Jacob straight line to OW-2 drawdown data
fit_ow2 = ptu.cooper_jacob_fit(
    ...,  # ← time array (hint: ow2['time_min'].values)
    ...,  # ← drawdown array (hint: ow2['drawdown_m'].values)
)

# ✏️ Plot the semi-log fit
ax = ptu.plot_semilog(
    ...,  # ← time array
    ...,  # ← drawdown array
    fit_ow2,
    well_id='OW-2',
    r_m=r_ow2,
)
plt.show()

# Read off the fitted slope
delta_s = fit_ow2['slope']
print(f"\nΔs (slope per log cycle): {delta_s:.3f} m")
print(f"t₀ (zero-drawdown time):  {fit_ow2['t0_min']:.3f} min")

In [ ]:
# Checkpoint: verify your slope value
check_task_with_solution('task05_pt_checkpoint_1')

**Step 2 — Calculate transmissivity $T$**

Apply the Cooper-Jacob formula to compute transmissivity from the slope:

$$T = \frac{2.3\, Q}{4\pi\, \Delta s}$$

**Available variables:** `Q` (pumping rate in m³/d), `delta_s` (slope from Step 1)

In [ ]:
# --- Step 2: Calculate transmissivity T for OW-2 ---
# Formula: T = 2.3 * Q / (4 * π * Δs)

T_ow2 = ...  # ✏️ write the formula using Q and delta_s

print(f"T = 2.3 × {Q:.0f} / (4π × {delta_s:.3f}) = {T_ow2:.1f} m²/d")

In [ ]:
# Checkpoint: verify your T value
check_task_with_solution('task05_pt_checkpoint_2')

**Step 3 — Convert to hydraulic conductivity $K$**

$$K = \frac{T}{b}$$

**Available variables:** `T_ow2` (from Step 2), `b` (aquifer thickness in m)

In [ ]:
# --- Step 3: Convert T to hydraulic conductivity K ---
# Formula: K = T / b

K_ow2 = ...  # ✏️ write the formula using T_ow2 and b

print(f"K = {T_ow2:.1f} / {b:.0f} = {K_ow2:.1f} m/d")
print(f"Compare with Notebook 4 uniform K = 20 m/d")

In [ ]:
# Checkpoint: verify your K value
check_task_with_solution('task05_pt_checkpoint_3')

**Step 4 — Extend to all wells**

If the Theis assumptions hold, all 4 observation wells should give the **same $T$** — because the Cooper-Jacob slope depends only on $Q$ and $T$, not on distance $r$. Use `ptu.analyze_all_wells()` to run the analysis on all wells at once, then `ptu.summarize_results()` to print a comparison table.

```python
# Function signatures:
ptu.analyze_all_wells(df, Q_m3d=..., b_m=...)  → (results_df, fits_dict)
ptu.summarize_results(results_df)               → prints table
```

In [ ]:
# --- Step 4: Extend analysis to all 4 observation wells ---

# ✏️ Run Cooper-Jacob on all wells
results_df, fits = ptu.analyze_all_wells(
    ...,      # ← pumping test dataframe
    Q_m3d=...,  # ← pumping rate [m³/d]
    b_m=...,    # ← aquifer thickness [m]
)

# ✏️ Print the results summary
ptu.summarize_results(...)  # ← pass the results dataframe

In [ ]:
# Semi-log plots for all 4 wells (visual check)
fig, axes = ptu.plot_all_wells_cj(pt_data, fits, results_df)
fig.suptitle('Cooper-Jacob Analysis — All Observation Wells', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Checkpoint: verify mean K across all wells
check_task_with_solution('task05_pt_checkpoint_4')

**Step 5 — Interpret the results**

The transmissivity estimates from the 4 wells are similar but not identical. Why might this be?

In [ ]:
# Checkpoint: why does T vary slightly across wells?
create_multiple_choice('task05_pt_checkpoint_5')

In [ ]:
# --- Store mean K for use in PEST++ calibration (Section 5) ---
K_mean = results_df['K_md'].mean()
K_pumping_test = K_mean

print(f"→ K from pumping test: {K_pumping_test:.1f} m/d (mean of {len(results_df)} wells)")

### 3.5 Interpretation

The consistency of $T$ across wells validates the analysis. The pumping test gives us $K \approx 26$ m/d in the central aquifer — somewhat higher than the uniform K = 20 m/d from Notebook 4.

> **Think about it:** If the PT-derived K differs from the uniform K value, what does that tell us about the spatial variability of K?

---

## 4 - Calibration Concepts

Before running the calibration, let's review the key ideas.

### 4.1 Calibration metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **ME** | $\frac{1}{n}\sum(h_{sim} - h_{obs})$ | Bias (positive = over-prediction) |
| **MAE** | $\frac{1}{n}\sum|h_{sim} - h_{obs}|$ | Average error magnitude |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(h_{sim} - h_{obs})^2}$ | Penalises large errors |
| **NRMSE** | $\frac{\text{RMSE}}{h_{max} - h_{min}} \times 100\%$ | RMSE relative to observed range |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Fraction of variance explained |

> **Targets for production models:** NRMSE < 10 %, R² > 0.9, ME ≈ 0, water balance error < 1 %
>
> **Realistic targets for this exercise:** With only 4 real wells, 5 noisy synthetic observations, a single-layer steady-state model, and ~20 pilot points, we cannot expect production-grade calibration. Here we aim for:
> - **RMSE improvement** over the uncalibrated model
> - **Water balance error** < 1 %
> - A **physically plausible** K field (no extreme artefacts)

### 4.2 Pilot points

Instead of calibrating a few large K zones, we estimate K at **distributed pilot points**:
- Pilot points are scattered across the active domain
- Each has its own adjustable K parameter (log-transformed)
- Values between points are **interpolated** (kriging or IDW) to create a smooth K field
- More flexible than zones, but needs **regularisation** to avoid overfitting

**Preferred-value regularisation:** Each pilot point has a soft constraint pulling it toward the initial K estimate (20 m/d). Without this, pilot points far from any observation well are unconstrained and can drift to physically unrealistic values. The regularisation weight controls the trade-off: too low and you get spatial artefacts; too high and the calibration cannot adjust K enough to fit the data.

### 4.3 Prior information

The pumping test gives us $K \approx 26$ m/d at a specific location. We encode this as a **prior information equation** in PEST++. During calibration, PEST++ balances fitting head observations against honouring this independent estimate.

### 4.4 PEST++ (pestpp-glm)

**PEST++** is the industry standard for model-independent parameter estimation. We use **pestpp-glm** (Gauss-Levenberg-Marquardt):
1. Run the model with current parameters
2. Compute sensitivities (Jacobian matrix)
3. Update parameters to reduce the objective function
4. Repeat until convergence

---

## 5 - PEST++ Calibration with Pilot Points

We set up and run PEST++ with:
- **~20 pilot points** distributed across the active domain
- **Observations:** AWEL mean-head measurements + synthetic observations
- **Prior information:** K from the pumping test
- **Parameter bounds:** 1–300 m/d (plausible for a gravel aquifer)
- **Preferred-value regularisation:** prevents unconstrained pilot points from drifting to extreme values

### 5.1 Install PEST++ (if needed)

In [ ]:
# --- 1. Check if pestpp-glm is available; install if needed ---
import shutil
if shutil.which('pestpp-glm') is None:
    print("pestpp-glm not found on PATH. Installing via get-pestpp...")
    import subprocess
    # get-pestpp is bundled with pyemu and downloads USGS PEST++ binaries
    venv_bin = os.path.join(os.path.dirname(sys.executable))
    result = subprocess.run(
        [sys.executable, '-m', 'pyemu.utils.get_pestpp',
         '--subset', 'pestpp-glm', venv_bin],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        # Try the get-pestpp CLI command directly
        result = subprocess.run(
            ['get-pestpp', '--subset', 'pestpp-glm', venv_bin],
            capture_output=True, text=True,
        )
    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
    if shutil.which('pestpp-glm'):
        print("pestpp-glm installed successfully.")
    else:
        print("Warning: pestpp-glm could not be installed automatically.")
        print("Install manually: get-pestpp /path/to/bin")
else:
    print(f"pestpp-glm found: {shutil.which('pestpp-glm')}")

### 5.2 Build the PEST++ setup

In [ ]:
# --- 2. Import PEST setup utilities ---
from setup_pest_calibration import (
    build_pest_setup,
    run_pestpp_glm,
    load_pest_results,
    get_calibrated_k_field,
    _interpolate_pp_to_grid,
    _nearest_pilot_point,
)

In [ ]:
# --- 3. Place the pumping test in the central aquifer ---
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters
active_mask = idomain.flatten() > 0

active_x = xc[active_mask]
active_y = yc[active_mask]
pt_x = float(np.median(active_x))
pt_y = float(np.median(active_y))
pt_location = (pt_x, pt_y)

# Fallback: if the pumping test exercise was skipped, use the known K value
if 'K_pumping_test' not in dir():
    K_pumping_test = 26.0  # Cooper-Jacob result (K_true = 26 m/d)
    print("(Using default K = 26 m/d — complete the pumping test exercise to derive this yourself)")

print(f"Pumping test location: ({pt_x:.0f}, {pt_y:.0f})")
print(f"K from pumping test:   {K_pumping_test:.1f} m/d")

In [ ]:
# --- 4. Build PEST++ calibration directory ---

# Identify which RIV cells belong to the Sihl (for leakance multiplier)
from shapely.geometry import Point

sihl_poly = river_gdf[river_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()
riv_data = riv.stress_period_data.get_data(0)

sihl_cell_ids = []
for rec in riv_data:
    cid = rec['cellid']
    cell_idx = cid[-1] if isinstance(cid, tuple) else cid
    pt = Point(xc[cell_idx], yc[cell_idx])
    if sihl_poly.contains(pt):
        sihl_cell_ids.append(cell_idx)
sihl_cell_ids = np.array(sihl_cell_ids)
print(f"Sihl RIV cells: {len(sihl_cell_ids)} out of {len(riv_data)} total")

pest_ws, pp_xy = build_pest_setup(
    model_ws=workspace,
    modelgrid=modelgrid,
    obs_df=obs_gdf,
    pt_K=K_pumping_test,
    pt_location=pt_location,
    n_pilot_points=20,
    k_initial=20.0,
    k_lower=1.0,
    k_upper=300.0,
    pt_weight=3.0,
    reg_weight=1.5,
    variogram_range=6000.0,
    idomain=idomain,
    sihl_cell_ids=sihl_cell_ids,
)

In [ ]:
# --- 5. Visualise pilot point locations ---
fig, ax = plt.subplots(figsize=(12, 8))

# Plot the active domain as background
active_arr = np.where(idomain.flatten() > 0, 1.0, np.nan)
quick_model_plot(modelgrid, active_arr, boundary_gdf=boundary_gdf,
                 cmap='Blues', colorbar_label='', ax=ax,
                 title='Pilot Points and Observation Network')

# Overlay pilot points and observation wells
ax.scatter(pp_xy[:, 0], pp_xy[:, 1], s=60, c='blue', marker='^',
           edgecolors='black', linewidth=0.8, zorder=4, label='Pilot points')

# Separate real and synthetic observations
real_obs = obs_gdf[~obs_gdf['is_synthetic']]
synth_obs = obs_gdf[obs_gdf['is_synthetic']]

if len(real_obs) > 0:
    ax.scatter(real_obs.geometry.x, real_obs.geometry.y,
               s=120, c='red', edgecolors='black', linewidth=1.2, zorder=5,
               label='AWEL wells')
if len(synth_obs) > 0:
    ax.scatter(synth_obs.geometry.x, synth_obs.geometry.y,
               s=80, c='#E67E22', marker='^', edgecolors='black', linewidth=1.0, zorder=5,
               label='Synthetic wells')

ax.scatter(*pt_location, s=200, c='gold', marker='*', edgecolors='black',
           linewidth=1.2, zorder=6, label='Pumping test site')

ax.legend(loc='lower left')
fig.tight_layout()
plt.show()

### 5.3 Run pestpp-glm

In [ ]:
# --- 6. Run PEST++ calibration ---
# This may take a few minutes depending on model size.
success = run_pestpp_glm(pest_ws)

if success:
    print("\nCalibration completed successfully.")
else:
    print("\nCalibration encountered issues — check the .rec file for details.")

### 5.4 Calibration results

In [ ]:
# --- 7. Load calibrated K field ---
try:
    k_field, pp_df = get_calibrated_k_field(pest_ws, modelgrid, variogram_range=6000.0)
    print("Calibrated K field loaded from PEST++ results.")
except Exception as e:
    print(f"Could not load PEST results ({e}). Using initial K field.")
    pp_log_vals = np.full(len(pp_xy), np.log10(20.0))
    k_field = _interpolate_pp_to_grid(
        pp_xy, pp_log_vals, modelgrid, method='idw', variogram_range=6000.0,
    )
    pp_df = pd.DataFrame({
        'x': pp_xy[:, 0], 'y': pp_xy[:, 1],
        'K_md': np.full(len(pp_xy), 20.0),
    })

In [ ]:
# --- 8. Visualise calibrated K field ---
fig, ax = plt.subplots(figsize=(12, 8))

# Mask inactive cells
k_plot = np.where(idomain.flatten() > 0, k_field.flatten(), np.nan)
quick_model_plot(modelgrid, k_plot, boundary_gdf=boundary_gdf,
                 cmap='viridis', colorbar_label='Hydraulic Conductivity K [m/d]',
                 title='Calibrated Hydraulic Conductivity Field', ax=ax)

# Overlay pilot points colored by K
sc = ax.scatter(pp_df['x'], pp_df['y'], c=pp_df['K_md'],
                cmap='viridis', s=50, edgecolors='black', linewidth=0.8,
                vmin=np.nanmin(k_plot), vmax=np.nanmax(k_plot), zorder=4)
ax.scatter(*pt_location, s=200, c='gold', marker='*', edgecolors='black',
           linewidth=1.2, zorder=6, label=f'PT site (K≈{K_pumping_test:.0f} m/d)')

ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

In [ ]:
# --- 9. Run calibrated model and compare metrics ---
# Update K in the NPF package
gwf.npf.k.set_data(k_field)

# Apply calibrated Sihl leakance multiplier
par_df, _ = load_pest_results(pest_ws)
sihl_mult = 1.0
if 'sihl_leakance_mult' in par_df.index:
    mult_log = par_df.loc['sihl_leakance_mult', 'optimised']
    sihl_mult = 10.0 ** mult_log
    print(f"Sihl leakance multiplier: {sihl_mult:.1f}\u00d7 "
          f"(log10 = {mult_log:.3f}, effective \u03bb = {1.3e-7 * sihl_mult:.2e} 1/s)")

    riv = gwf.get_package('RIV')
    spd = riv.stress_period_data.get_data(0)
    sihl_set = set(sihl_cell_ids.tolist())
    for i in range(len(spd)):
        cid = spd[i]['cellid']
        cell_idx = cid[-1] if isinstance(cid, tuple) else cid
        if cell_idx in sihl_set:
            spd[i]['cond'] *= sihl_mult
    riv.stress_period_data.set_data(spd, key=0)

sim.write_simulation()
success_cal, _ = sim.run_simulation(silent=True)

if success_cal:
    head_cal = gwf.output.head().get_data()

    sim_heads_cal = cu.extract_heads_at_observations(head_cal, obs_gdf, modelgrid)
    obs_gdf['sim_head_calibrated'] = sim_heads_cal

    valid_cal = ~np.isnan(sim_heads_cal)
    obs_cal = obs_gdf[valid_cal].copy()

    metrics_calibrated = cu.calculate_calibration_metrics(
        obs_cal['head_m'].values, obs_cal['sim_head_calibrated'].values,
    )

    print(cu.compare_metrics(metrics_initial, metrics_calibrated,
                             labels=("Uncalibrated", "Calibrated")))
else:
    print("Calibrated model run failed.")
    head_cal = head

In [ ]:
# Checkpoint 3: What is your calibrated RMSE (m)?
check_task_with_solution('task05_checkpoint_3')

In [ ]:
# --- 10. Before/after scatter plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

obs_cal = obs_gdf[~np.isnan(obs_gdf['simulated_head']) & ~np.isnan(obs_gdf.get('sim_head_calibrated', np.nan))].copy()

# Before calibration
ax = axes[0]
ax.scatter(obs_cal['head_m'], obs_cal['simulated_head'],
           c=['#E67E22' if s else '#2E86AB' for s in obs_cal['is_synthetic']],
           s=80, edgecolor='black', linewidth=0.5, zorder=3)
rng = [obs_cal['head_m'].min() - 1, obs_cal['head_m'].max() + 1]
ax.plot(rng, rng, 'k--', linewidth=1, zorder=1)
ax.set_xlim(rng); ax.set_ylim(rng)
ax.set_aspect('equal')
ax.set_xlabel('Observed Head (m a.s.l.)')
ax.set_ylabel('Simulated Head (m a.s.l.)')
ax.set_title('Before Calibration')
ax.grid(True, alpha=0.3)
ax.text(0.95, 0.05, f"RMSE = {metrics_initial['RMSE']:.2f} m\nR² = {metrics_initial['R2']:.3f}",
        transform=ax.transAxes, fontsize=10, va='bottom', ha='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# After calibration
if 'sim_head_calibrated' in obs_cal.columns:
    ax = axes[1]
    ax.scatter(obs_cal['head_m'], obs_cal['sim_head_calibrated'],
               c=['#E67E22' if s else '#2E86AB' for s in obs_cal['is_synthetic']],
               s=80, edgecolor='black', linewidth=0.5, zorder=3)
    ax.plot(rng, rng, 'k--', linewidth=1, zorder=1)
    ax.set_xlim(rng); ax.set_ylim(rng)
    ax.set_aspect('equal')
    ax.set_xlabel('Observed Head (m a.s.l.)')
    ax.set_ylabel('Simulated Head (m a.s.l.)')
    ax.set_title('After Calibration')
    ax.grid(True, alpha=0.3)
    ax.text(0.95, 0.05, f"RMSE = {metrics_calibrated['RMSE']:.2f} m\nR² = {metrics_calibrated['R2']:.3f}",
            transform=ax.transAxes, fontsize=10, va='bottom', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.tight_layout()
plt.show()

> **How good is "good enough"?** The calibrated model shows modest improvement over the uncalibrated model. This is expected: we have only 9 observations (5 of which are synthetic with substantial noise), a single-layer steady-state model, and ~20 pilot points. A production model would need many more observations, transient calibration, and possibly a multi-layer setup. For this teaching exercise, the key takeaway is understanding the **calibration process** — not achieving a perfect fit.

### 5.5 Prior information check

In [ ]:
# --- 11. Check PT K match ---
pp_pt_idx = _nearest_pilot_point(pp_xy, pt_location)
K_at_pt = pp_df.iloc[pp_pt_idx]['K_md'] if 'K_md' in pp_df.columns else 20.0

print(f"K from pumping test:              {K_pumping_test:.1f} m/d")
print(f"Calibrated K at nearest PP ({pp_pt_idx:02d}):  {K_at_pt:.1f} m/d")
print(f"Difference:                       {abs(K_at_pt - K_pumping_test):.1f} m/d "
      f"({abs(K_at_pt - K_pumping_test)/K_pumping_test*100:.0f}%)")

# Sihl leakance multiplier summary
if sihl_mult != 1.0:
    print(f"\nSihl leakance multiplier:         {sihl_mult:.1f}\u00d7")
    print(f"Effective Sihl \u03bb:                {1.3e-7 * sihl_mult:.2e} 1/s "
          f"(base: 1.3e-7 1/s)")

> **Why doesn't the PT K match exactly?** The prior information equation pulls the nearest pilot point toward the PT value, but it competes with the head observations. PEST++ finds a compromise: if nearby observations require a different K to fit the heads, the PT prior can be overridden — especially when observation weights collectively outweigh the single prior constraint. This illustrates a fundamental trade-off in inverse modelling between **data fit** and **parameter plausibility**.

---

## 6 - Calibration Assessment

### 6.1 Water balance verification

In [ ]:
# --- 12. Water balance ---
try:
    budget_df = format_budget_summary(gwf, sim)
    display(budget_df)

    # Check water balance error
    pct_error = budget_df.loc['PERCENT ERROR', 'Net (m3/d)']
    if isinstance(pct_error, (int, float)) and pct_error < 1.0:
        print("\n  Water balance error < 1 % — PASS")
    elif isinstance(pct_error, (int, float)):
        print(f"\n  Water balance error = {pct_error:.4f} % — review model setup")
except Exception as e:
    print(f"Could not read water balance: {e}")

In [ ]:
# Checkpoint 4: What is the water balance error (%)?
check_task_with_solution('task05_checkpoint_4')

### 6.2 Residual map

In [ ]:
# --- 13. Residual map — calibrated model ---
if 'sim_head_calibrated' in obs_gdf.columns:
    obs_cal = obs_gdf[~np.isnan(obs_gdf['sim_head_calibrated'])].copy()
    residuals_cal = obs_cal['sim_head_calibrated'].values - obs_cal['head_m'].values
    fig = cu.plot_residual_map(
        obs_cal, residuals_cal,
        boundary_gdf=boundary_gdf,
        title='Residual Map — Calibrated Model',
    )
    plt.show()

In [ ]:
# Checkpoint 5: Conceptual question on residual interpretation
create_multiple_choice('task05_checkpoint_5')

### Comprehension Check

> **Why is one pumping-test estimate of K particularly valuable when we already have head observations at four wells?**

<details>
<summary>Click to check your answer</summary>

The four AWEL wells are all clustered in one area and constrain the model only locally. The pumping test provides an **independent, physically-based** measurement of K at a different location — this breaks the spatial non-uniqueness and prevents the calibration from assigning unrealistic K values where no head data exists.

</details>

---

## Summary

> **What we accomplished**
>
> - **Assessed** our observation network and found it severely limited (few real wells, one cluster)
> - **Supplemented** with synthetic observations for teaching purposes
> - **Performed** Cooper-Jacob analysis on a pumping test → independent K ≈ 26 m/d
> - **Calibrated** the model with PEST++ using pilot points and prior information
> - **Verified** the calibrated model through metrics comparison and water balance

**Key insight:** Calibration is only as good as the data that constrains it. With a few head observations and 1 pumping test, the calibrated K field is *one plausible solution* — not *the* solution. More observation data would dramatically reduce non-uniqueness.

---

## What You're Taking Forward

You now have a calibrated MODFLOW 6 model with:
- A spatially varying K field (from pilot-point calibration)
- Documented calibration metrics and water balance
- An understanding of the **limitations** and **non-uniqueness** inherent in this calibration

The biggest remaining question: **how reliable are the model's predictions?**

---

## Next Steps

**Continue to:** [6_validation.ipynb](6_validation.ipynb) — where we test the calibrated model against independent data not used in calibration.

After that: [7_sensitivity_uncertainty.ipynb](7_sensitivity_uncertainty.ipynb) — where we explore how parameter uncertainty propagates to predictions.

## References

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling*. Academic Press.
- Cooper, H.H., Jacob, C.E. (1946). A generalized graphical method for evaluating formation constants and summarizing well-field history. *Trans. AGU*, 27(4), 526–534.
- Doherty, J. (2015). *Calibration and Uncertainty Analysis for Complex Environmental Models*. Watermark Numerical Computing.
- Fienen, M.N., Muffels, C.T., Hunt, R.J. (2009). On constraining pilot point calibration with regularization in PEST. *Ground Water*, 47(6), 835–844.
- White, J.T., et al. (2020). Towards improved environmental modeling outcomes. *Environ. Modelling & Software*, 131, 104749.